In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import TensorDataset,DataLoader
import torchvision.transforms as transforms

# Image scale and normalization

In [ ]:
transfrom = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
]
)


In [ ]:
trainset =  CIFAR10(root="/content/data",train=True,download=True,transform=transfrom)
testset = CIFAR10(root="/content/data",train=False,download=True,transform=transfrom)

In [ ]:
trainset

In [ ]:
testset

In [ ]:
from torch.utils.data.dataset import Dataset
train_loader = DataLoader(trainset,batch_size = 128, shuffle=True)
test_loader = DataLoader(testset,batch_size=128)

# Build the CNN

In [ ]:
class cnn(nn.Module):
  def __init__(self):
    super(cnn,self).__init__()

    self.convo_layers = nn.Sequential(

        nn.Conv2d(3,32,kernel_size=2, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2), # kernel_size and stride value

        nn.Conv2d(32,64,kernel_size=2, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),  # kernel_size and stride value

        nn.Conv2d(64,128,kernel_size=2, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)  # kernel_size and stride value
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128, 256),
        nn.ReLU(),

        nn.Linear(256,10)
    )

  def forward(self,x):
    x = self.convo_layers(x)
    x = x.view(x.size(0),-1)  # x value flatten
    x = self.fc_layers(x)

    return x

In [ ]:
model = cnn()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN model

In [ ]:
epochs = 10

for epoch in range(epochs):
  training_loss = 0.0

  for images, labels in train_loader:
    # forward propagation
    optimizer.zero_grad()
    output = model.forward(images)
    loss = criterion(output,labels)

    # backward propagation
    loss.backward()

    optimizer.step()  # update the parameters

    training_loss = training_loss + loss.item()

  print(f"Epoch : {epoch+1}/{epochs}, Train Loss : {training_loss/len(train_loader)}")

# Evaluate the model

In [ ]:
total =  0
correct =  0

model.eval()

with torch.no_grad():
  for images,labels in test_loader:
    outputs = model.forward(images)
    _,predicted = torch.max(outputs,1)

    correct += (predicted == labels).sum().item()
    total += labels.size(0)

print(f"Accuracy : {correct/total * 100}")